# Tác hại của rerank theo từng query: có dấu vs không dấu (Colab GPU)

Chạy lại bước 2 của `eval_bm25_fold.py` (dev 500, `none` và `add`, mỗi query hai bản có dấu / không dấu) và **ghi từng query**: hạng đáp án ở hybrid và sau rerank, top-10 hai bên, điểm cross-encoder. Sau đó `analyze_rerank_damage.py` in phân bố tác hại, bảng chéo có dấu × không dấu, và các query bị hại riêng do dấu nặng nhất.

Chỉ **đo và mô tả**. Không chọn quy tắc nào ở notebook này.

**Trước khi chạy**
1. *Runtime → Change runtime type → T4 GPU*.
2. Chuẩn bị `colab/fold_bundle.zip` (bản mới, có `analyze_rerank_damage.py`).
3. **Dữ liệu sẽ rời máy sang Google Colab.** Văn bản đã scrub nhưng vẫn còn tên service, mã lỗi, luồng nghiệp vụ.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "sentence-transformers==6.0.1"

In [ ]:
from google.colab import files
up = files.upload()            # chon fold_bundle.zip
BUNDLE = next(iter(up))
!unzip -oq {BUNDLE} -d /content/work
%cd /content/work
!ls pipeline data/processed

## 1. Chạy và ghi từng query

Kiểm tra nhanh: `goc keyword none` R@10 phải là **0,576**; `khong_dau hybrid+rerank add` phải gần **0,566** (lần Colab trước).

In [ ]:
!python pipeline/eval_bm25_fold.py --stage 2 --variants none add --device cuda \
    --out data/processed/bm25_fold_stage2_rerun.json \
    --dump-per-query data/processed/bm25_fold_perquery.jsonl

## 2. Phân tích tác hại

In [ ]:
!python pipeline/analyze_rerank_damage.py --dump data/processed/bm25_fold_perquery.jsonl --fold add --top 15

## 3. Tải kết quả về

Giải nén vào `D:/viettel/semantic/` rồi báo lại.

In [ ]:
!zip -q damage_results.zip data/processed/bm25_fold_perquery.jsonl data/processed/rerank_damage_add.json data/processed/bm25_fold_stage2_rerun.json
files.download("damage_results.zip")